# Listed directory Plant_Seg
Listed directory Data_exploration
Listed directory data
Listed directory archive
Listed directory plantsegv2
Viewed Metadatav2.csv:1-800
Ran command: `python -c "import json; f = open('f:/PyTorch_GPU/Plant_Seg/Data_exploration/data/archive/plantsegv2/coco_annotations.json'); data = json.load(f); print('Keys:', data.keys()); print('Info:', data.get('info', {})); print('Licenses:', len(data.get('licenses', []))); print('Categories:', data.get('categories', [])); print('Images sample:', data.get('images', [])[:1]); print('Annotations sample:', data.get('annotations', [])[:1]);"`

I have explored the nested folders and analyzed the files you requested. Here is a breakdown of the structure and an explanation of the dataset curation based on those two files.

### 1. Structure of the `data` directory
The directory is structured as follows:
*   `Data_exploration/data/`
    *   **`2409.pdf`**: This is likely a research paper or documentation associated with the dataset.
    *   **`archive.zip`**: A compressed 1.7GB archive containing the dataset.
    *   **`archive/plantsegv2/`**: The extracted contents of the zip file, which includes:
        *   **`images/`**: A folder containing the raw source images (e.g., `.jpg` files) of the plant leaves.
        *   **`annotations/`**: A folder containing the image-based masks (e.g., `.png` files) highlighting the diseased areas.
        *   **`Metadatav2.csv`**: Detailed tabular metadata for the entire dataset.
        *   **`coco_annotations.json`**: Polygonal annotation data formatted in the standard COCO format.

---

### 2. How the Dataset is Curated
The dataset is an image segmentation and classification dataset for plant pathology. It was curated by collecting images of diseased plant leaves from a wide variety of online sources (news articles, forums, academic papers, stock photos, and agricultural sites). Once the images were sourced:
1.  They were categorized by the type of **Plant** (e.g., Apple, Banana) and the specific **Disease** (e.g., *apple black rot*, *banana anthracnose*).
2.  Pixel-level **segmentation masks** were generated to identify precisely where the disease occurs on the leaf.
3.  The images were randomly separated into standard machine learning splits (**Training, Validation, and presumably Test**) to be used for training computer vision models.
4.  The annotations were stored both as image masks (for semantic segmentation) and converted into COCO polygon formats (for instance segmentation/object detection).

---

### 3. Understanding `Metadatav2.csv`
This file acts as a comprehensive index for every image in the dataset. It provides rich context mapping the raw images to their labels. The key columns are:
*   **`Name`**: The filename of the raw image (e.g., `apple_black_rot_1.jpg`).
*   **`Index` & `Plant` & `Disease`**: The numeric class index, the plant species, and the specific disease label.
*   **`Resolution`**: The dimensions of the original image (e.g., `640x480`).
*   **`Label file`**: The filename of the corresponding black-and-white mask image (e.g., `apple_black_rot_1.png`) located in the annotations folder.
*   **`Mask ratio`**: A float value (e.g., `0.0404...`) indicating what percentage of the image consists of the diseased area. 
*   **`URL`**: The exact website link from where the image was scraped during curation.
*   **`Split`**: Indicates whether the image belongs to the `Training`, `Validation`, or `Test` set.

### 4. Understanding `coco_annotations.json`
This is a large (~168 MB) JSON file that structures the dataset into the standard **COCO (Common Objects in Context)** format, which is the industry standard for training object detection and instance segmentation models (like Mask R-CNN or YOLO).

It contains several main dictionaries:
*   **`info`**: General metadata (`{'year': 2023, 'version': '1.0', 'description': 'PlantSeg'}`).
*   **`categories`**: Maps every disease to a unique ID (e.g., `{'id': 1, 'name': 'apple black rot'}`).
*   **`images`**: An array of all the images, assigning them a unique `id` and defining their `width` and `height`.
*   **`annotations`**: The actual labeling data. For every single diseased spot identified in an image, this array provides:
    *   `category_id`: Which disease it is.
    *   `image_id`: Which image it belongs to.
    *   `bbox`: The bounding box coordinates `[x, y, width, height]` wrapping the disease.
    *   `segmentation`: An array of `[x, y]` coordinates mapping out the exact polygon boundary of the diseased area.
    *   `area`: The pixel area of the bounding box/polygon.

In summary, the `Metadatav2.csv` is best for exploring the dataset metadata (URLs, splits, mask sizes), while the `coco_annotations.json` is ready to be directly plugged into an AI model training pipeline.

# Data Pipeline Analysis & Visualization

This section performs an in-depth exploratory data analysis (EDA) of the PlantSeg dataset, exploring `Metadatav2.csv` and `coco_annotations.json`.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import cv2
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Paths
base_dir = r"f:\PyTorch_GPU\Plant_Seg\Data_exploration\data\archive\plantsegv2"
csv_path = os.path.join(base_dir, "Metadatav2.csv")
json_path = os.path.join(base_dir, "coco_annotations.json")
images_dir = os.path.join(base_dir, "images")
annotations_dir = os.path.join(base_dir, "annotations")

# Load data
df = pd.read_csv(csv_path)
with open(json_path, "r") as f:
    coco_data = json.load(f)

print("Data loaded successfully!")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\amans\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\amans\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "f:\PyTorch_GPU\torch_gpu\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "f:\PyTorch_GPU\torch_gpu\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
 

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## 1. Dataset Splits and Distribution
Let's visualize how the dataset is split across Training, Validation, and Testing phases.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df, x="Split", palette="viridis", order=df["Split"].value_counts().index
)
plt.title("Dataset Split Distribution")
plt.xlabel("Split")
plt.ylabel("Number of Images")
for p in plt.gca().patches:
    plt.gca().annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
    )
plt.show()

## 2. Plant and Disease Distributions
Understanding which plants and diseases are most prevalent in the dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plant distribution
sns.countplot(
    data=df,
    y="Plant",
    palette="muted",
    order=df["Plant"].value_counts().index[:15],
    ax=axes[0],
)
axes[0].set_title("Top 15 Plants by Image Count")
axes[0].set_xlabel("Count")

# Disease distribution
sns.countplot(
    data=df,
    y="Disease",
    palette="rocket",
    order=df["Disease"].value_counts().index[:15],
    ax=axes[1],
)
axes[1].set_title("Top 15 Diseases by Image Count")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

## 3. Mask Ratio Distribution
The Mask Ratio indicates what percentage of the image is covered by the disease mask. This tells us if we're dealing with small localized spots or large infected areas.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["Mask ratio"].dropna(), bins=50, kde=True, color="coral")
plt.title("Distribution of Mask Ratios")
plt.xlabel("Mask Ratio (0.0 to 1.0)")
plt.ylabel("Frequency")
plt.show()

## 4. Image Resolution Analytics
Understanding image dimensions is crucial for model resizing pipelines.

In [ ]:
# Extract widths and heights from resolution
df[["Width", "Height"]] = df["Resolution"].str.split("x", expand=True).astype(float)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Width", y="Height", alpha=0.5, color="teal")
plt.title("Image Resolutions (Width vs Height)")
plt.xlabel("Width (pixels)")
plt.ylabel("Height (pixels)")
plt.show()

print("\nTop 5 Resolutions:")
print(df["Resolution"].value_counts().head(5))

## 5. COCO Annotations Deep Dive
The COCO JSON contains instance-level annotations. Let's analyze bounding box areas.

In [ ]:
annotations = coco_data.get("annotations", [])
areas = [ann.get("area", 0) for ann in annotations if "area" in ann]

plt.figure(figsize=(10, 6))
sns.histplot(areas, bins=50, log_scale=(False, True), color="purple")
plt.title("Distribution of COCO Bounding Box Areas (Log Scale Y)")
plt.xlabel("Area (pixels squared)")
plt.ylabel("Frequency (Log Scale)")
plt.show()

## 6. Real-Time Data Visualization
Visualizing an actual image along with its COCO polygon segmentation and bounding box.

In [ ]:
from matplotlib.patches import Polygon, Rectangle
import random

# Pick a random image with annotations
random.seed(42)  # for reproducibility
sample_ann = random.choice(annotations)
sample_img_info = next(
    img for img in coco_data["images"] if img["id"] == sample_ann["image_id"]
)
cat_info = next(
    cat for cat in coco_data["categories"] if cat["id"] == sample_ann["category_id"]
)

img_path = os.path.join(images_dir, sample_img_info["file_name"])

if os.path.exists(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)

    # Draw Bounding Box
    bbox = sample_ann["bbox"]
    rect = Rectangle(
        (bbox[0], bbox[1]),
        bbox[2],
        bbox[3],
        linewidth=2,
        edgecolor="red",
        facecolor="none",
    )
    ax.add_patch(rect)

    # Draw Segmentation Polygon
    for seg in sample_ann["segmentation"]:
        poly = np.array(seg).reshape((int(len(seg) / 2), 2))
        patch = Polygon(poly, closed=True, fill=True, color="cyan", alpha=0.4)
        ax.add_patch(patch)

    plt.title(f"Image: {sample_img_info['file_name']} | Category: {cat_info['name']}")
    plt.axis("off")
    plt.show()
else:
    print(f"Image not found at {img_path}. (Skipping visualization)")